In [ ]:
from openai import OpenAI

openai_client = OpenAI()


In [ ]:
from gitsource import GithubRepositoryDataReader, chunk_documents

In [ ]:
reader = GithubRepositoryDataReader(repo_owner='evidentlyai', repo_name='docs', allowed_extensions={"md", "mdx"},)

In [ ]:
files = reader.read()

In [ ]:
print(files)

In [ ]:
pip install minisearch

In [ ]:
from minsearch import AppendableIndex

In [ ]:
parsed_docs= [doc.parse() for doc in files ]

In [ ]:
print(parsed_docs[15])

In [ ]:
chunked_docs= chunk_documents(parsed_docs, size=3000, step=1500)

In [ ]:
chunked_docs[2]

In [ ]:
index= AppendableIndex(
       text_fields=["title","content","description"],
       keyword_fields=["filename"]
)

In [ ]:
index.fit(chunked_docs)

In [ ]:
def search(query):
    results = index.search(query=query,num_results=5)
    return results

In [ ]:
import json

RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

In [ ]:
question =  "How do I create a dahsbord in Evidently?"
search_results= search(question)
user_prompt = build_prompt(question,search_results)
print(user_prompt)

In [ ]:
messages =[{"role":"system","content":RAG_INSTRUCTIONS},
           {"role":"user","content":user_prompt}
          ]

In [ ]:
response = openai_client.responses.create(model='gpt-4o-mini',input=messages)

In [ ]:
print(response.output_text)


In [ ]:
instruction = """You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Use only facts from the knowledge base when answering.
IMPORTANT:  you cannot find the answer, inform the user."""

search_tool = { "type":"function", 
               "name": "search", 
               "description": "Search the documentation knowledge base for relevant information.",
                 "parameters": 
                 {"type": "object", 
                  "properties": {
                      "query": {
                      "type": "string", 
                      "description": "The search query to lookup in the index."
                      }
                      },
                      "required": [
                          "query"
                          ]}}

In [ ]:
messages =[{
    "role":"system","content":instruction
    },
    {
        "role":"user","content":question
        }
        
              ]

In [ ]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool])

In [ ]:
response.usage.input_tokens



In [ ]:
tool_call = response.output[0]
tool_call

In [ ]:
messages.append(tool_call)

In [ ]:
tool_call.arguments

In [ ]:
arguments = json.loads(tool_call.arguments)
arguments

In [ ]:
search_results = search(query='create dashboard in Evidently')

In [ ]:
search_results = search(**arguments)
search_results[:1]

In [ ]:
call_output = {
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": json.dumps(search_results),
}

In [ ]:
messages.append(call_output)
messages

In [ ]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

In [ ]:
print(response.output_text)

In [ ]:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    else: 
        result = 'not found tool "{name}"'

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }
    

In [ ]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

IMPORTANT: When you explore the knowledge base, make at least 3 different
searchers to make sure you explore the topic well.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [ ]:
question = "How do I create a dahsbord in Evidently?"

In [ ]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

In [ ]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool],
)

In [ ]:
print(response)

In [ ]:
message_history.extend(response.output)
message_history

In [ ]:
for message in response.output:
    if message.type == 'function_call':
        print(f'executing {message.name}({message.arguments})...')
        tool_call_output = make_call(message)
        message_history.append(tool_call_output)

In [ ]:
len(message_history)
message_history

In [ ]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool],
)

response.usage.input_tokens


In [ ]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [ ]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=[search_tool],
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

In [ ]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": [
            "filename",
            "title",
            "description",
            "content"
        ]
    }
}

In [ ]:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    elif name == 'add_entry':
        result = add_entry(**arguments)
    else: 
        result = 'not found tool "{name}"'

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

In [ ]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]
tools = [search_tool, add_entry_tool]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

In [ ]:
message_history.append(
    {"role": "user", "content": "add this content to our database"}
)

In [ ]:
while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

In [ ]:
index.docs[-1]